# Nebula Qwen2.5-Coder 7B Unified QLoRA

Use a GPU runtime. Upload one staged `nebula-100k-*-colab-bundle.zip`. Checkpoints are written to Google Drive, and rerunning the training cell resumes from the latest checkpoint. Start with 5K; do not jump directly to 90K.


In [ ]:
from google.colab import drive, files
from pathlib import Path
import shutil, zipfile

drive.mount('/content/drive')
uploaded = files.upload()
bundle_name = next((name for name in uploaded if name.endswith('.zip')), None)
if not bundle_name:
    raise RuntimeError('Upload the Nebula 100K Colab bundle zip.')
workspace = Path('/content/nebula-100k')
shutil.rmtree(workspace, ignore_errors=True)
workspace.mkdir(parents=True)
with zipfile.ZipFile(bundle_name) as archive:
    archive.extractall(workspace)
%cd /content/nebula-100k
print('Workspace restored:', workspace)


In [ ]:
!pip uninstall -y -q torchao torchvision || true
!pip install -q -r training/requirements-qlora.txt
import torch, bitsandbytes, transformers, peft
print('PyTorch:', torch.__version__, 'CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('bitsandbytes:', bitsandbytes.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before continuing.')


In [ ]:
from pathlib import Path
import json

stage_files = sorted(Path('training/nebula-100k/stages').glob('train-*.jsonl'))
validation_files = sorted(Path('training/nebula-100k/stages').glob('validation-*.jsonl'))
if len(stage_files) != 1 or len(validation_files) != 1:
    raise RuntimeError(f'Expected one train and validation stage, found {stage_files} and {validation_files}')
train_file, validation_file = stage_files[0], validation_files[0]
train_count = sum(1 for _ in train_file.open(encoding='utf-8'))
validation_count = sum(1 for _ in validation_file.open(encoding='utf-8'))
print('Train:', train_file, train_count)
print('Validation:', validation_file, validation_count)
print(json.loads(Path('training/nebula-100k/manifest.json').read_text())['claim'])


In [ ]:
from pathlib import Path
import subprocess, sys

stage_size = int(train_file.stem.split('-')[-1])
output = Path('/content/drive/MyDrive/NebulaTraining') / f'nebula-qwen2.5-coder-7b-{stage_size}-v1' / 'adapter'
output.mkdir(parents=True, exist_ok=True)
print('Checkpoints:', output)
subprocess.run([
    sys.executable, 'training/scripts/train_qlora.py',
    '--config', 'training/configs/qwen2.5-coder-7b-nebula-100k-qlora.json',
    '--train-file', str(train_file),
    '--validation-file', str(validation_file),
    '--output-dir', str(output),
    '--resume',
], check=True)


In [ ]:
from pathlib import Path
adapter = output
required = ['adapter_config.json', 'adapter_model.safetensors', 'nebula-training-report.json']
missing = [name for name in required if not (adapter / name).exists()]
if missing:
    raise RuntimeError(f'Training did not produce final adapter files: {missing}')
print('Training complete:', adapter)
print((adapter / 'nebula-training-report.json').read_text()[:3000])


## Next gate

Do not merge immediately. First evaluate the adapter against the committed coding, stress, and cyber suites, inspect raw failures, and compare it with the base model. A completed training cell is not proof that the model improved.


In [ ]:
from google.colab import files
import shutil
archive = shutil.make_archive(f'/content/nebula-qwen2.5-coder-7b-{stage_size}-adapter', 'zip', adapter)
print('Adapter archive:', archive)
files.download(archive)
